In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os


import pyrootutils

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_XLA_FLAGS", "--tf_xla_auto_jit=0")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
import tensorflow as tf

tf.get_logger().setLevel('ERROR')

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)


In [ ]:
from building.scaling import (
    ScalingRunConfig,
    load_full_arrays,
    run_experiments,
    summarize_results,
    plot_summary,
)

In [ ]:
COLLECTION = "diff_genus"
N_SAMPLES = 10
EPOCHS = 50
PATIENCE = 10
BATCH_SIZE = 32
SEED = 42
THRESHOLD = 0.5
BUILD_MODEL = "sincnet"
MODELS_DIR = ROOT / "models" / BUILD_MODEL
RESULTS_FILE = MODELS_DIR / "results.jsonl"

In [ ]:
config = ScalingRunConfig(
    collection=COLLECTION,
    build_model=BUILD_MODEL,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    threshold=THRESHOLD,
    models_dir=MODELS_DIR,
    results_file=RESULTS_FILE,
)

catalog = load_full_arrays(
    collection=COLLECTION,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

print(f"Loaded {len(catalog.class_names)} classes:")
print(catalog.class_names)

In [ ]:

baseline_rows = run_experiments(
    catalog,
    config,
    run_baseline=True,
    run_scaling=False,
)
print(f"New baseline runs: {len(baseline_rows)}")
if baseline_rows:
    print("Last baseline row:")
    print(baseline_rows[-1])

In [ ]:
from building.scaling import print_baselines
print_baselines(catalog, RESULTS_FILE)

In [ ]:
scaling_rows = run_experiments(
    catalog,
    config=config,
    n_samples=N_SAMPLES,
    k_values=range(2, len(catalog.class_names)),
    run_baseline=False,
    run_scaling=True,
)
print(f"New scaling runs: {len(scaling_rows)}")
if scaling_rows:
    print("Last scaling row:")
    print(scaling_rows[-1])

In [ ]:
baseline_metrics, summary_df = summarize_results(RESULTS_FILE)
print(f"Baseline recall: {baseline_metrics.recall:.4f}")
print(f"Baseline precision: {baseline_metrics.precision:.4f}")
summary_df

In [ ]:
plot_summary(summary_df, baseline=baseline_metrics)